# MiniMax H3 加速模型 · 魔搭 ModelScope / PAI-DSW 版

**下载只走一条通道：ModelScope SDK（`modelscope.snapshot_download`）**，不依赖 Kaggle Dataset、不依赖 HuggingFace / hf-mirror、不需要代理。用到的文件在魔搭站内全部有镜像，内网实测 85–185 MB/s。

## 一、本版下载设计

| 关注点 | 做法 |
|---|---|
| 通道 | `modelscope.snapshot_download` 单通道 |
| 粒度 | `allow_patterns=[单个文件]`，不整仓拉 100+ GB |
| 落盘 | 每个文件独立暂存目录 → 软链到 `ComfyUI/models/{diffusion_models,vae,text_encoders,loras}` |
| 校验 | 下完按「大小 ≥ 标称 90%」判定，不达标重试（最多 3 次，间隔递增） |
| 持久化 | `/mnt/workspace` 是 NAS，实例关闭后文件保留，重开零下载 |
| GPU 镜像 | 社区实测默认镜像 `ubuntu22.04-cuda13.0.3-py312-torch2.13.0`，正好是 `int8_convrot` 要的 cu130 |
| 访问 | cloudflared 隧道（免费 Notebook 网关不支持 8188 自定义端口） |

已核实的站内镜像（路径与 HF 原名一致，本 Notebook 就按这些拉）：

| 仓库 | 用到什么 |
|---|---|
| `MATLOWAI/minimax-h3-fused-turbo-int8-convrot` | 20.98 GB 主模型 + 全部工作流 json |
| `Kijai/MiniMax-H3-experimental` | 3.17 GB 视频 VAE、轻量 VAE、加速 LoRA、w4a8 轻量主模型 |
| `Comfy-Org/MiniMax-H3` | 官方 NVFP4 文本编码器 15.69 GB、音频 VAE 0.61 GB、w6a8 / fp8 量化版 |

> **越狱编码器**（`Momoking/Qwen3-VL-32B-Heretic-MiniMax-H3-NVFP4`）魔搭没有镜像，本版删掉了 HF 回退通道，所以默认用官方编码器。真要用：自己把 15.68 GB 的文件放到 `/mnt/workspace/minimax_h3/_download/te_heretic/` 或 `ComfyUI/models/text_encoders/`，第 5 单元会自动扫到并复用，再把 `TEXT_ENCODER` 改成 `"heretic"`（工作流文件名会自动改写）。

## 二、显存需求（重要）

文章标题的「8GB 显存可跑」不是这套 21 GB fused turbo 文件的实测。作者 README 的计时全部来自 **单张 RTX PRO 6000 96 GB、权重全驻留**：baked fused 单文件 **峰值 47.8 GB / 常驻 42 GB**（等价的 live-LoRA 路径 68.9 GB / 64 GB）。小卡靠 ComfyUI 按需 load/offload transformer、CLIP、VAE 也能跑，只是慢。

| 显存 | 做法 |
|---|---|
| ≥ 48 GB | 默认工作流全速，1152×640 × 243 帧 / 4 步约 76 s |
| 24 GB（A10） | `--cache-none --reserve-vram 1` + `workflows/lowvram/`（FFN 分块）+ 降到 0.3–0.7 MP、≤124 帧 |
| 16 GB | 上面全部 + `--novram`，明显变慢 |
| 8 GB | 换 `w4a8` / `w6a8` / GGUF 轻量版 + 重度 offload，**配 ComfyUI 内置 H3 模板**而非本套工作流 |

> **双显卡 / 合计显存**：ComfyUI `main.py` 单进程只用 `cuda:0`，多卡合计不参与判档。例如 T4 ×2 合计 32 GB，仍按 GPU0 的 16 GB 判为 `low` 档并自动加 `--novram --cache-none`；按 32 GB 判会漏掉 `--novram` 反而更容易 OOM。另起实例用第二张卡时设 `COMFY_GPU_ID=1` 并换端口；也可设 `VRAM_TIER="low"/"mid"/"high"` 手动覆盖。显存探测失败一律按 `low` 保守处理。

> **`--lowvram` 已过时**：ComfyUI v0.45.3 默认开启动态显存（DynamicVRAM），`--lowvram` 的官方说明是 "Doesn't do anything if dynamic vram is enabled"。改用 `--cache-none` / `--reserve-vram` / `--novram`，本 Notebook 按检测到的显存自动选。

## 三、断点续跑 / 复用已有文件

`/mnt/workspace` 是持久化 NAS，实例暂停或到期后文件仍在。第 5 单元会把下列位置全扫一遍，**文件齐全就直接复用，绝不重下**：

- `minimax_h3/_download/{key}/`（本 Notebook 自己的暂存，下载中断的现场）
- `minimax_h3/download**/`、`minimax_h3/models/`（历史尝试留下的目录）
- `minimax_h3/ComfyUI/models/**`、`/mnt/workspace/ComfyUI/models/**`、`/mnt/workspace/models/**`
- `~/.cache/modelscope/hub`（SDK 自己的缓存，含无扩展名的 blob）

判定标准是**文件名 + 尺寸 ≥ 目标 90%**；没复用时打印「同名但太小 / 尺寸吻合但改名」的候选，最终决策写进 `minimax_h3/output/cache_scan.json`，下次能直接看出为什么没复用。所以下载中断后重新 Run All，只会补缺的那几个文件，已下好的 20 GB 主模型不会再动。

## 四、魔搭专属的坑（本 Notebook 已规避）

1. **路径嵌套**：`modelscope download --local-dir models/diffusion_models` 会在下面再建一层 `diffusion_models/`，ComfyUI 下拉框里就找不到模型。本版统一下到独立暂存目录，再按 `models/{diffusion_models,vae,text_encoders,loras}` 软链归位。
2. **新增模型必须重启 ComfyUI**：它只在启动阶段扫描模型目录，刷新网页无效。
3. **访问端口**：DSW / JupyterLab 可把地址栏 `/lab` 改成 `/proxy/8188/`；但**魔搭免费 Notebook 网关不支持 8188 等自定义端口**（会 302 到 `modelscope.cn/error`，报错 -30），官方推荐 **cloudflared 隧道**。本版两种都给，默认起 cloudflared。
4. **免费实例**：空闲约 1 小时自动关闭、单次约 8 小时强制停；模型一定放 `/mnt/workspace`。
5. **量化内核**：`int8_convrot` VAE 需 ComfyUI ≥ 0.36.0；NVFP4 / INT8 ConvRot 走 comfy-kitchen 的 Triton 后端，启动要加 `--enable-triton-backend`（默认关闭）。


In [ ]:
# ---------- 1. 环境自检 ----------
import os, sys, json, shutil, subprocess, platform, urllib.request, urllib.parse, time, re
from pathlib import Path

WORKSPACE = Path("/mnt/workspace") if Path("/mnt/workspace").is_dir() else Path.cwd()
print("Python       :", sys.version.split()[0])
print("平台         :", platform.platform())
print("持久化目录   :", WORKSPACE, "(可写)" if os.access(WORKSPACE, os.W_OK) else "(不可写!)")

for p in ["/", str(WORKSPACE)]:
    q = Path(p)
    if q.exists():
        total, used, free = shutil.disk_usage(q)
        note = "  ⚠ NAS 上报值，非真实配额" if free / 1e9 > 1e5 else ""
        print(f"磁盘 {p:22s} 可用 {free/1e9:7.1f} GB / 共 {total/1e9:7.1f} GB{note}")

MEM_GB = 0.0
try:
    kv = {}
    for line in open("/proc/meminfo"):
        if ":" in line:
            k, v = line.split(":", 1)
            kv[k] = float(v.split()[0]) * 1024
    MEM_GB = kv["MemTotal"] / 1e9
    print(f"系统内存     : {MEM_GB:.1f} GB / 可用 {kv['MemAvailable']/1e9:.1f} GB")
except Exception:
    print("系统内存     : 无法读取")

try:
    print("GPU          :", subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        text=True, stderr=subprocess.DEVNULL).strip().replace("\n", " | "))
except Exception:
    print("GPU          : 未检测到 nvidia-smi（纯下载不需要；AMD 机型看 rocm-smi）")

VRAM_GB = 0.0
SM_MAJOR = 0
GPU_COUNT, VRAM_TOTAL_GB = 0, 0.0
try:
    import torch
    print("PyTorch/CUDA :", torch.__version__, "/", torch.version.cuda)
    if torch.cuda.is_available():
        GPU_COUNT = torch.cuda.device_count()
        VRAM_TOTAL_GB = sum(torch.cuda.get_device_properties(i).total_memory
                            for i in range(GPU_COUNT)) / 1e9
        pr = torch.cuda.get_device_properties(0)
        VRAM_GB = pr.total_memory / 1e9
        SM_MAJOR = pr.major
        print(f"GPU0         : {pr.name}  {VRAM_GB:.1f} GB  (SM {pr.major}.{pr.minor})")
        print(f"GPU 数量     : {GPU_COUNT} 张 | 合计 {VRAM_TOTAL_GB:.1f} GB")
        if GPU_COUNT > 1:
            print(f"  ⚠ ComfyUI 单进程只用一张卡，判档按 GPU0 的 {VRAM_GB:.1f} GB")
        if VRAM_GB >= 48:
            tier = "全速档：可全驻留，直接跑默认工作流"
        elif VRAM_GB >= 24:
            tier = "A10 档：lowvram 工作流 + cache-none + 降分辨率"
        elif VRAM_GB >= 16:
            tier = "吃紧：加 --novram + 0.3MP/124帧，或改轻量模型"
        else:
            tier = "不建议跑 21GB fused，改 w4a8/w6a8/GGUF + 官方内置模板"
        print("显存档位     :", tier)
        if MEM_GB and MEM_GB < 45:
            print(f"⚠ 系统内存仅 {MEM_GB:.0f} GB，权重约 40 GB offload 时可能 CPU OOM")
        if SM_MAJOR < 8:
            print(f"⚠ SM {pr.major}.{pr.minor} 无 FP4 硬件路径，量化内核可能回退慢速")
except ImportError:
    print("PyTorch      : 未安装（默认镜像应预装，装 ComfyUI 时会带上）")

try:
    import modelscope
    print("modelscope   :", getattr(modelscope, "__version__", "已安装"))
    try:
        from modelscope.hub.constants import MODELSCOPE_CACHE
        print("SDK 缓存目录 :", MODELSCOPE_CACHE)
    except Exception:
        pass
except ImportError:
    print("modelscope   : 未安装 -> 下一个单元会装")

for name, url in [("ModelScope", "https://www.modelscope.cn"),
                  ("GitHub", "https://github.com")]:
    try:
        urllib.request.urlopen(url, timeout=8)
        print(f"网络 {name:12s}: OK")
    except Exception as e:
        print(f"网络 {name:12s}: 不可达 ({type(e).__name__})")

In [ ]:
# ---------- 2. 依赖（只补 modelscope SDK，不动预装的 torch / CUDA） ----------
import subprocess, sys, inspect

def _ver(mod):
    try:
        m = __import__(mod)
        return getattr(m, "__version__", "已安装")
    except ImportError:
        return None

_v = _ver("modelscope")
if _v is None:
    # DSW 镜像通常已预装；只在缺失时安装，避免升级破坏镜像里别的包
    print("未装 modelscope，开始安装 ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "modelscope"])
    _v = _ver("modelscope")
print("modelscope :", _v)

# 提前暴露能力差异：老 SDK 没有 allow_patterns 就只能整仓下载，必须在配置前说清楚
from modelscope import snapshot_download
_P = inspect.signature(snapshot_download).parameters
print("snapshot_download 参数 :", ", ".join(_P))
_FILTER = next((n for n in ("allow_patterns", "allow_file_pattern", "allow_file_patterns") if n in _P), None)
_LOCAL = "local_dir" if "local_dir" in _P else ("cache_dir" if "cache_dir" in _P else None)
print("按文件过滤   :", _FILTER or "不支持（需 pip install -U modelscope）")
print("落盘目录参数 :", _LOCAL or "不支持")
if _FILTER is None or _LOCAL is None:
    raise RuntimeError("modelscope SDK 版本过旧，无法只下单个文件；请先 pip install -U modelscope")


## 四、配置区

| 变量 | 说明 |
|---|---|
| `PLAN` | `workflows` / `minimal`(3.8 GB) / `core`(24.8 GB) / `full`(≈40.4 GB) |
| `TEXT_ENCODER` | `official`（默认，工作流原生文件名）/ `heretic`（魔搭无镜像，需自备文件）/ `none` |
| `LIGHT_MODEL` | `None`，或 `w4a8`(12.5 GB) / `w6a8`(16.0 GB) / `fp8`(21.0 GB)。**这些配 ComfyUI 内置 H3 模板，不是本套 MATLOWAI 工作流** |
| `EXTRA_VAE` | 追加轻量 VAE / 加速 LoRA |
| `MS_REVISION` | 魔搭仓库分支，默认 `master` |
| `MS_WORKERS` | SDK 并发下载线程，默认 16；小机器可降到 8 |
| `GIT_PROXY` | GitHub clone 慢时填代理前缀，例如 `https://ghfast.top/`，自行确认可用性 |
| `INSTALL_COMFYUI` / `START_COMFYUI` | 只想搬模型就设 False |

首次建议 `PLAN="minimal"` 验证通路，再改 `full`。


In [ ]:
# ---------- 3. 配置 ----------
PLAN = "full"
TEXT_ENCODER = "official"       # official / heretic（需自备文件）/ none
LIGHT_MODEL = None              # None / "w4a8" / "w6a8" / "fp8"
EXTRA_VAE = []                  # 例如 ["light_vae", "lora_4step"]

MS_REVISION = "master"          # 魔搭仓库分支
MS_WORKERS = 16                 # snapshot_download 并发线程
GIT_PROXY = "https://gh-proxy.com"   # GitHub 直连不通时用代理前缀；置空则走直连

ROOT = WORKSPACE / "minimax_h3"          # /mnt/workspace 持久化，实例关闭也保留
STAGING = ROOT / "_download"
MODEL_ROOT = ROOT / "ComfyUI" / "models"
OUT = ROOT / "output"
COMFY_DIR = ROOT / "ComfyUI"
COMFY_TAG = "v0.45.3"
COMFY_PORT = 8188
INSTALL_COMFYUI = True
START_COMFYUI = True
START_TUNNEL = True
TUNNEL_PROTOCOL = "http2"
COMFY_EXTRA_ARGS = []           # 例如 ["--fast-disk"]（仅当 /mnt/workspace 是高速本地盘时）
GPU_VENDOR = "amd"              # "nvidia" / "amd"；决定启动参数与环境变量

for d in [ROOT, STAGING, MODEL_ROOT, OUT]:
    d.mkdir(parents=True, exist_ok=True)
print("模型/暂存 :", ROOT)
print("GPU 厂商  :", GPU_VENDOR)
print("工作流输出 :", OUT)
print("下载通道   : modelscope SDK（单通道，无 HF / 无 Kaggle）")


In [ ]:
# ---------- 4. 资源清单（全部来自魔搭站内镜像，路径与 HF 原名一致）----------
# AMD GPU 兼容版：全 BF16 / FP16 / FP32，避开 NVIDIA 专属的 INT8 ConvRot 和 NVFP4 AWQ
# 参考：魔搭社区 AMD MI300X 192GB 部署经验（ROCm / PyTorch+rocm）
#   - INT8 ConvRot 主模型：NVIDIA CUDA 专用，AMD 上无加速内核 -> 改 BF16 pruned
#   - NVFP4 AWQ 编码器：Blackwell 专属量化，AMD 只能软件模拟（慢 ~10x） -> 改 BF16
#   - 视频 VAE：改 FP16（音频 VAE FP32 已通用，保持）
MS_MAIN  = "Comfy-Org/MiniMax-H3"
MS_ORG   = "Comfy-Org/MiniMax-H3"
MS_MOTION = "MATLOWAI/MiniMax-H3-Motion-Adapter"

RESOURCES = {
    "main_model": dict(
        ms=MS_MAIN,
        file="diffusion_models/minimax_h3_fl2va_pruned_bf16.safetensors",
        dst="diffusion_models", gb=40.23, desc="主模型 BF16 pruned，AMD 兼容"),
    "video_vae": dict(
        ms=MS_ORG,
        file="vae/minimax_h3_video_vae_fp16.safetensors",
        dst="vae", gb=5.21, desc="视频 VAE FP16，AMD 兼容"),
    "audio_vae": dict(
        ms=MS_ORG,
        file="vae/minimax_h3_audio_vae_fp32.safetensors",
        dst="vae", gb=0.61, desc="音频 VAE FP32，工作流必需"),
    "te_heretic": dict(
        ms=None,
        file="qwen3vl_32b_heretic_minimax_h3_nvfp4.safetensors",
        dst="text_encoders", gb=15.68, desc="越狱版：魔搭无镜像，需自备文件（不推荐 AMD）"),
    "te_official": dict(
        ms=MS_ORG,
        file="text_encoders/qwen3vl_32b_minimax_h3_bf16.safetensors",
        dst="text_encoders", gb=51.51, desc="官方 BF16，工作流默认文件名（AMD 兼容）"),
}

EXTRAS = {
    "light_vae":     dict(ms=MS_ORG, file="minimax_h3_lightvae_lite_vae_int8_convrot.safetensors", dst="vae", gb=1.04, desc="轻量 VAE（INT8，AMD 慎用）"),
    "light_encoder": dict(ms=MS_ORG, file="minimax_h3_lightvae_lite_encoder_fp16.safetensors", dst="vae", gb=0.20, desc="轻量 VAE encoder FP16"),
    "lynnreal_vae":  dict(ms=MS_ORG, file="minimax_h3_lynnreal_light_vae_int8_convrot.safetensors", dst="vae", gb=2.14, desc="剪枝 light VAE（INT8，AMD 慎用）"),
    "lora_4step":    dict(ms=MS_ORG, file="loras/minimax_h3_pdmd_4step_lora_avg_rank_57_bf16.safetensors", dst="loras", gb=0.56, desc="PDMD 4 步 LoRA BF16"),
    "lora_8step":    dict(ms=MS_ORG, file="loras/minimax_h3_DMAD_8step_full_lora_avg_rank_58_bf16.safetensors", dst="loras", gb=0.56, desc="DMAD 8 步 LoRA BF16"),
    "lora_flashgen": dict(ms=MS_ORG, file="loras/minimax_h3_4step_lora_flashgen_v1.0_768p_fl2va_pruned_avg_rank_13_bf16.safetensors", dst="loras", gb=0.19, desc="4 步 flashgen LoRA BF16"),
    "lora_temporal": dict(ms=MS_MOTION, file="temporal_expansion/minimax_h3_temporal_expansion_step375_r16.safetensors", dst="loras", gb=0.063, desc="temporal expansion 375 步"),
    "light_w4a8":    dict(ms=MS_ORG, file="minimax_h3_fl2va_pruned_w4a8_mixed.safetensors", dst="diffusion_models", gb=12.54, desc="轻量主模型 w4a8"),
    "light_w6a8":    dict(ms=MS_ORG, file="diffusion_models/minimax_h3_fl2va_pruned_w6a8.safetensors", dst="diffusion_models", gb=15.98, desc="轻量主模型 w6a8"),
    "light_fp8":     dict(ms=MS_ORG, file="diffusion_models/minimax_h3_fl2va_pruned_fp8_scaled.safetensors", dst="diffusion_models", gb=20.96, desc="fp8 主模型"),
}
ALL = {**RESOURCES, **EXTRAS}
TE_KEY = {"heretic": "te_heretic", "official": "te_official", "none": None}[TEXT_ENCODER]
LIGHT_KEY = {None: None, "w4a8": "light_w4a8", "w6a8": "light_w6a8", "fp8": "light_fp8"}[LIGHT_MODEL]

PLANS = {
    "workflows": [],
    "minimal":   ["video_vae", "audio_vae"],
    "core":      ["video_vae", "audio_vae", "main_model"],
    "full":      ["video_vae", "audio_vae", "main_model", "lora_temporal"] + ([TE_KEY] if TE_KEY else []),
}
_base = PLANS[PLAN] + [k for k in EXTRA_VAE if k in EXTRAS]
if LIGHT_KEY:
    _base.append(LIGHT_KEY)
    if TE_KEY:
        _base.append(TE_KEY)
SELECTED = list(dict.fromkeys(_base))
print(f"PLAN={PLAN} / TEXT_ENCODER={TEXT_ENCODER} / LIGHT_MODEL={LIGHT_MODEL}")
print(f"计划 {len(SELECTED)} 个文件，合计约 {sum(ALL[k]['gb'] for k in SELECTED):.2f} GB")
for k in SELECTED:
    it = ALL[k]
    print(f"  {k:15s} {it['gb']:6.2f} GB  {it['ms'] or '需自备':32s} {Path(it['file']).name}")
if LIGHT_KEY:
    print("⚠ 轻量主模型请用 ComfyUI 内置 H3 模板，MATLOWAI 工作流引用的是 fused 文件")
_no_ms = [k for k in SELECTED if not ALL[k].get("ms")]
if _no_ms:
    print(f"⚠ {_no_ms} 在魔搭无镜像：只有本机已有完整文件时才会复用，否则第 7 单元（执行下载）会直接报错，不会先白下 40 GB")


## 五、下载：只走 ModelScope SDK

- **唯一通道** `modelscope.snapshot_download(model_id=..., revision=MS_REVISION, local_dir=..., allow_patterns=[单个文件])`，按文件过滤，不整仓下载。
- **暂存隔离**：每个文件下到 `minimax_h3/_download/{key}/`，下完再软链到 `ComfyUI/models/{dst}/`，天然避开魔搭 CLI 的路径嵌套坑。
- **重试与校验**：失败最多 3 次（间隔递增），下完按「大小 ≥ 标称 90%」判定，不达标重下；成功时打印实测 MB/s。
- **复用**：暂存里已有完整同名文件直接跳过，实例重启后零下载。


In [ ]:
# ---------- 5. 扫描已有缓存，决定谁要下载 ----------
import json
# 只要能在这台机器上找到「完整」的文件就不再下载。扫描范围放宽到历史尝试留下的目录 +
# modelscope SDK 自己的缓存（实例重启后 /mnt/workspace 保留，缓存自然命中）。
CACHE_ROOTS = [
    STAGING, MODEL_ROOT, ROOT, OUT,
    ROOT / "download", ROOT / "downloads", ROOT / "models",
    WORKSPACE / "ComfyUI" / "models", WORKSPACE / "models", WORKSPACE / "downloads",
    Path.home() / ".cache" / "modelscope" / "hub",
]
SUFFIX = (".safetensors", ".gguf", ".bin", ".pt")

def _indexable(p):
    # SDK 缓存里的 blob 没有扩展名，单独放行，否则缓存里的 15 GB 编码器永远扫不到
    return p.suffix in SUFFIX or p.parent.name == "blobs"

def build_index():
    idx, big, seen = {}, [], set()
    for root in CACHE_ROOTS:
        if not root.exists():
            continue
        for p in root.rglob("*"):
            try:
                if not p.is_file() or not _indexable(p):
                    continue
                key = str(p.resolve())          # 多个扫描根会互相重叠，去重
                if key in seen:
                    continue
                seen.add(key)
                size = p.stat().st_size
                idx.setdefault(p.name, []).append((size, p))
                if size > 1e9:
                    big.append((size, p))
            except OSError:
                continue
    return idx, big

IDX, BIG = build_index()
print(f"索引：{len(IDX)} 个文件名 / {len(BIG)} 个 >1GB 文件（扫描根 {sum(1 for r in CACHE_ROOTS if r.exists())} 个）")

def find_cached(item):
    name = Path(item["file"]).name
    need = item["gb"] * 1e9 * 0.90            # 允许 10% 误差，防四舍五入
    hits = [(s, p) for s, p in IDX.get(name, []) if s >= need]
    return max(hits, key=lambda t: t[0])[1] if hits else None

def diagnose(item):
    """没复用时，把「差一点就能复用」的候选打出来，避免以为脚本没扫。"""
    name = Path(item["file"]).name
    exact = item["gb"] * 1e9
    tol = max(exact * 0.02, 50e6)
    for s, q in IDX.get(name, []):
        print(f"        同名但太小: {s/1e9:6.2f} / {item['gb']:.2f} GB  {q}")
    for s, q in BIG:
        if abs(s - exact) <= tol and q.name != name:
            print(f"        尺寸吻合但改名: {s/1e9:6.2f} GB  {q}")

resolved, to_download = {}, []
for key in SELECTED:
    it = ALL[key]
    p = find_cached(it)
    if p:
        resolved[key] = dict(path=p, source="cache")
        print(f"  REUSE {Path(it['file']).name[:52]:52s} <- {p}")
        continue
    to_download.append(key)
    print(f"  FETCH {Path(it['file']).name[:52]:52s} ~{it['gb']} GB")
    diagnose(it)

print(f"\n复用 {len(resolved)} 个 / 待下载 {len(to_download)} 个，"
      f"预计 {sum(ALL[k]['gb'] for k in to_download):.2f} GB")
free = shutil.disk_usage(WORKSPACE).free / 1e9
print(f"{WORKSPACE} 可用 {free:.1f} GB" +
      ("（NAS 上报值，非真实配额，请以实例规格为准）" if free > 1e5 else ""))
# 落到盘上，方便下一次直接看「上次为什么没复用」
(OUT / "cache_scan.json").write_text(json.dumps({
    "reuse": {k: str(v["path"]) for k, v in resolved.items()},
    "fetch": to_download,
    "roots": [str(r) for r in CACHE_ROOTS if r.exists()],
}, ensure_ascii=False, indent=2), encoding="utf-8")


In [ ]:
# ---------- 6. 下载：ModelScope SDK 单通道 ----------
import inspect, time
from modelscope import snapshot_download as ms_download

_P = inspect.signature(ms_download).parameters
_FILTER = next((n for n in ("allow_patterns", "allow_file_pattern", "allow_file_patterns") if n in _P), None)
_LOCAL = "local_dir" if "local_dir" in _P else ("cache_dir" if "cache_dir" in _P else None)
if _FILTER is None or _LOCAL is None:
    raise RuntimeError("modelscope SDK 版本过旧（缺 %s），请先 pip install -U modelscope"
                       % ("按文件过滤" if _FILTER is None else _LOCAL))

def _ok_size(p, item):
    return p is not None and p.exists() and p.stat().st_size >= item["gb"] * 1e9 * 0.90

def locate(folder, item):
    name = Path(item["file"]).name
    hits = [p for p in Path(folder).rglob(name) if p.is_file()]
    return max(hits, key=lambda p: p.stat().st_size) if hits else None

def dl_modelscope(item, folder):
    """只拉 item['file'] 这一个文件，落到 folder（保留仓库内相对路径）。"""
    kw = {"model_id": item["ms"]}
    if "revision" in _P:
        kw["revision"] = MS_REVISION
    if "max_workers" in _P:
        kw["max_workers"] = int(globals().get("MS_WORKERS", 8))
    if "quiet" in _P:
        kw["quiet"] = False
    kw[_LOCAL] = str(folder)
    kw[_FILTER] = [item["file"]]
    return ms_download(**kw)

def fetch(key):
    it = ALL[key]
    if not it.get("ms"):
        raise RuntimeError(
            f"{key} 在魔搭没有镜像（{Path(it['file']).name}）。本版只走 modelscope SDK，"
            "请把该文件手动放到 " + str(STAGING / key) + " 或 ComfyUI/models/text_encoders/ 后重跑第 5 单元，"
            "或把 TEXT_ENCODER 改回 'official'")
    folder = STAGING / key
    folder.mkdir(parents=True, exist_ok=True)
    cached = locate(folder, it)
    if _ok_size(cached, it):
        print(f"  [{key}] 暂存已完整，跳过")
        return cached, "resume"

    last = None
    for attempt in range(1, 4):
        t0 = time.time()
        try:
            print(f"  [{key}] modelscope 第 {attempt} 次 ... ({it['ms']})")
            dl_modelscope(it, folder)
            p = locate(folder, it)
            if _ok_size(p, it):
                dt = max(time.time() - t0, 1e-6)
                sz = p.stat().st_size / 1e9
                print(f"  [{key}] OK {sz:.2f} GB  {sz*1e3/dt:6.1f} MB/s  ({dt:.0f}s)")
                return p, "modelscope"
            print(f"  [{key}] 文件不完整（{(p.stat().st_size/1e9) if p else 0:.2f} / {it['gb']} GB），重试")
        except Exception as e:
            last = e
            print(f"  [{key}] 失败: {type(e).__name__}: {str(e)[:200]}")
        time.sleep(5 * attempt)
    raise RuntimeError(f"{key} 下载失败；最后一次错误: {last}")


In [ ]:
# ---------- 7. 执行下载 ----------
# 先校验：魔搭没镜像的文件，绝不等 40 GB 下完才报错
_blocked = [k for k in to_download if not ALL[k].get("ms")]
if _blocked:
    raise RuntimeError(
        "这些文件在魔搭没有镜像，无法用 modelscope SDK 下载：" +
        "、".join(f"{k}({Path(ALL[k]['file']).name})" for k in _blocked) +
        "。请手动放入对应暂存目录后重跑第 5 单元，或改 TEXT_ENCODER='official'")

if not to_download:
    print("无需下载，全部复用已有文件")
newly_downloaded = {}
t_all = time.time()
for key in to_download:
    it = ALL[key]
    print(f"\n=== {key}  ~{it['gb']} GB  ({it['ms'] or '需自备'}) ===")
    p, src = fetch(key)
    resolved[key] = dict(path=p, source=src)
    newly_downloaded[key] = p
print(f"\n完成：复用 {len(SELECTED)-len(newly_downloaded)} 个，新下载 {len(newly_downloaded)} 个，"
      f"耗时 {time.time()-t_all:.0f}s")
for k, p in newly_downloaded.items():
    print(f"  {k:15s} {p.stat().st_size/1e9:6.2f} GB  {p}")


In [ ]:
# ---------- 8. 工作流：从魔搭镜像仓库取 json + 越狱编码器改写 ----------
WF_API = "https://www.modelscope.cn/api/v1/models/{repo}/repo?Revision=master&FilePath={path}"
wf_dir = OUT / "workflows"
wf_dir.mkdir(parents=True, exist_ok=True)

# 文件树来自魔搭 API（Recursive=True），只取 workflows 下的 json
try:
    tree_url = f"https://www.modelscope.cn/api/v1/models/{MS_MAIN}/repo/files?Revision=master&Recursive=True"
    tree = json.load(urllib.request.urlopen(tree_url, timeout=30))
    wf_files = [f.get("Path") or f.get("Name") for f in tree["Data"]["Files"]
                if (f.get("Path") or f.get("Name", "")).startswith("workflows/")
                and (f.get("Path") or f.get("Name", "")).endswith(".json")]
except Exception as e:
    print("文件树获取失败，改用固定清单:", e)
    wf_files = []

if not wf_files:
    wf_files = ["workflows/01_reference_4step_sla.json",
                "workflows/lowvram/01_reference_4step_sla_lowvram.json"]

# 已有工作流缓存（历史运行输出 / 手工放置）优先复用，避免重复下载
WF_CACHE_ROOTS = [
    WORKSPACE / "minimax_h3_out" / "workflows",   # 早期运行留下的输出目录
    WORKSPACE / "workflows",
    WORKSPACE / "minimax_h3" / "workflows",
]

def _find_wf_cache(rel):
    sub = Path(rel).relative_to("workflows")
    for root in WF_CACHE_ROOTS:
        if not root.exists():
            continue
        hit = root / sub
        if hit.is_file() and hit.stat().st_size > 100 and \
           hit.read_bytes().lstrip().startswith((b"{", b"[")):
            return hit
        # 缓存是扁平目录时按文件名兜底
        flat = [p for p in root.rglob(sub.name) if p.is_file() and p.stat().st_size > 100]
        if flat:
            return max(flat, key=lambda p: p.stat().st_size)
    return None

n_reuse = n_copy = n_fetch = 0
for rel in wf_files:
    target = wf_dir / Path(rel).relative_to("workflows")
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and target.stat().st_size > 100:
        n_reuse += 1
        continue
    hit = _find_wf_cache(rel)
    if hit:
        shutil.copy2(hit, target)
        n_copy += 1
        continue
    try:
        url = WF_API.format(repo=MS_MAIN, path=urllib.parse.quote(rel, safe="/"))
        data = urllib.request.urlopen(url, timeout=30).read()
        if not data.lstrip().startswith((b"{", b"[")):
            print("  非 JSON 内容，跳过", rel)
            continue
        target.write_bytes(data)
        n_fetch += 1
    except Exception as e:
        print("  下载失败", rel, type(e).__name__)
print(f"工作流 json: {n_reuse + n_copy + n_fetch} 个 -> {wf_dir}")
print(f"  本地已有 {n_reuse} / 缓存复用 {n_copy} / 新下载 {n_fetch}")

# 越狱编码器文件名与工作流里写死的官方名不同，生成一份改写副本
if TEXT_ENCODER == "heretic":
    official = Path(ALL["te_official"]["file"]).name
    heretic = Path(ALL["te_heretic"]["file"]).name
    dst_dir = OUT / "workflows_heretic"
    n = 0
    for src in wf_dir.rglob("*.json"):
        txt = src.read_text(encoding="utf-8", errors="ignore")
        if official in txt:
            t = dst_dir / src.relative_to(wf_dir)
            t.parent.mkdir(parents=True, exist_ok=True)
            t.write_text(txt.replace(official, heretic), encoding="utf-8")
            n += 1
    print(f"越狱版工作流 {n} 个 -> {dst_dir}（用这个目录的 json，别用 workflows/）")
else:
    print("TEXT_ENCODER=official，工作流文件名无需改写")

In [ ]:
# ---------- 9. 软链归位 + manifest + 依赖缺口扫描 ----------
import urllib.parse

for sub in ["diffusion_models", "vae", "text_encoders", "loras"]:
    (MODEL_ROOT / sub).mkdir(parents=True, exist_ok=True)

for key, info in resolved.items():
    it = ALL[key]
    src = Path(info["path"]).resolve()
    dst = MODEL_ROOT / it["dst"] / Path(it["file"]).name
    if dst.exists() or dst.is_symlink():
        if dst.is_symlink() or dst.resolve() != src:
            dst.unlink()
    if not dst.exists():
        try:
            dst.symlink_to(src)
        except OSError:
            shutil.copy2(src, dst)
    print(f"  {it['dst']:17s} {dst.name[:58]:58s} <- {info['source']}")

manifest = {
    "env": "modelscope / pai-dsw",
    "download_channel": "modelscope.snapshot_download",
    "workspace": str(WORKSPACE),
    "model_root": str(MODEL_ROOT),
    "workflow_root": str(OUT / "workflows"),
    "plan": PLAN, "text_encoder": TEXT_ENCODER,
    "files": [
        {"key": k, "name": Path(ALL[k]["file"]).name, "gb": ALL[k]["gb"],
         "ms_repo": ALL[k]["ms"], "ms_revision": MS_REVISION,
         "file": ALL[k]["file"], "path": str(Path(info["path"]).resolve()),
         "source": info["source"]}
        for k, info in resolved.items()
    ],
}
(OUT / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

present = {p.name for p in MODEL_ROOT.rglob("*.safetensors")}
refs = set()
for f in (OUT / "workflows").rglob("*.json"):
    refs |= set(re.findall(r"[A-Za-z0-9_.\-]+\.safetensors", f.read_text(encoding="utf-8", errors="ignore")))
missing = sorted(r for r in refs if r not in present)
print("\n本地模型文件:", len(present))
for r in sorted(present):
    print("   ", r)
print("工作流引用但本地还缺:", missing or "无")
print("\nmanifest:", OUT / "manifest.json")

In [ ]:
# ---------- 9b. 接通机器上已有的 ComfyUI（幂等，可反复运行） ----------
# 本 Notebook 自带的 ComfyUI 在 ROOT/ComfyUI，与 MODEL_ROOT 同目录，天然接通。
# 若机器上另有 ComfyUI（例如 DSW 上 /mnt/workspace/ComfyUI 属于别的项目），
# 这里写一份 extra_model_paths.yaml 让它也能看到 H3 模型和工作流，不复制/移动任何权重。
import shutil

LINK_EXTERNAL = True
EXTERNAL_COMFY_ROOTS = [WORKSPACE / "ComfyUI", Path("/mnt/data/ComfyUI")]

self_comfy = (ROOT / "ComfyUI").resolve()
candidates = [p for p in EXTERNAL_COMFY_ROOTS
              if (p / "main.py").exists() and p.resolve() != self_comfy]

if not LINK_EXTERNAL:
    print("LINK_EXTERNAL=False，跳过接通")
elif not candidates:
    print("未发现外部 ComfyUI；本 Notebook 自带的 ComfyUI 与 models/ 同目录，天然接通。")
else:
    yaml_block = "\n".join([
        "",
        "# MiniMax H3 model root - managed by minimax_h3_modelscope.ipynb",
        "# 新增模型会自动出现在 ComfyUI 下拉框，无需重新挂软链",
        "minimax_h3:",
        f"    base_path: {MODEL_ROOT}/",
        "    diffusion_models: diffusion_models",
        "    vae: vae",
        "    text_encoders: text_encoders",
        "    loras: loras",
        "",
    ])
    for comfy in candidates:
        yml = comfy / "extra_model_paths.yaml"
        old = yml.read_text(encoding="utf-8") if yml.exists() else ""
        if "minimax_h3:" in old:
            # 已由本 Notebook 管理：整段替换，保持幂等
            head = old.split("# MiniMax H3 model root")[0].rstrip()
            yml.write_text(head + yaml_block, encoding="utf-8")
            action = "updated"
        else:
            # 别的项目可能已有配置，只追加自己的段，绝不覆盖
            yml.write_text(old.rstrip() + yaml_block, encoding="utf-8")
            action = "appended" if old.strip() else "created"
        print(f"[{action}] {yml}")
        try:
            import yaml as _yaml
            cfg = _yaml.safe_load(yml.read_text(encoding="utf-8"))
            print("  yaml 解析 OK ->", cfg.get("minimax_h3"))
        except ImportError:
            print("  (PyYAML 未安装，跳过解析校验；ComfyUI 启动时会自行加载)")

        wf_src = OUT / "workflows"
        wf_dst = comfy / "user" / "default" / "workflows" / "minimax_h3"
        made = kept = 0
        if wf_src.exists():
            for src in sorted(wf_src.rglob("*.json")):
                if src.name.endswith(".api.json"):
                    continue
                dst = wf_dst / src.relative_to(wf_src)
                dst.parent.mkdir(parents=True, exist_ok=True)
                if dst.is_symlink() and dst.resolve() == src.resolve():
                    kept += 1
                    continue
                if dst.exists() or dst.is_symlink():
                    continue
                try:
                    dst.symlink_to(src.resolve())
                except OSError:
                    shutil.copy2(src, dst)
                made += 1
        print(f"  工作流: 新建 {made} / 已存在 {kept} -> {wf_dst}")

    print("\n接通完成。重启那份 ComfyUI 后生效（新增模型也需重启，刷新网页无效）。")
    print("验证：启动日志应出现 'Adding extra search path ... minimax_h3/ComfyUI/models/...'")

## 六、安装 ComfyUI + 插件（可跳过）

- 装到 `/mnt/workspace/minimax_h3/ComfyUI`，合并源码时**跳过上游 `models/`**，前面软链好的模型原样保留，免配 `extra_model_paths.yaml`。
- 若机器上**已有**另一套 ComfyUI（如 `/mnt/workspace/ComfyUI` 属于别的项目），上一单元已自动写好 `extra_model_paths.yaml` 并把工作流挂进它的 UI 目录，**不复制权重、不影响原有模型**。
- 插件 requirements 自动过滤 `torch` / `torchaudio` / `numpy` 等行，避免把魔搭预装的 cu130 / PyTorch 2.13 环境降级。
- `comfy-kitchen`（INT8 ConvRot / NVFP4 内核）已在 ComfyUI requirements 里，启动时用 `--enable-triton-backend` 打开 Triton 后端。
- GitHub clone 慢或失败：设 `GIT_PROXY = "https://ghfast.top/"` 之类的前缀后重跑本单元（第三方代理可用性请自行确认）。

> 这一段要在 **GPU 实例**里跑，首次安装约 5–10 分钟。只想搬模型就设 `INSTALL_COMFYUI = False`。

In [ ]:
# ---------- 10. 安装 ComfyUI 本体 ----------
if not INSTALL_COMFYUI:
    print("跳过 ComfyUI 安装")
else:
    if not (COMFY_DIR / "main.py").exists():
        tmp = ROOT / "_comfyui_git"
        if tmp.exists():
            shutil.rmtree(tmp)
        url = GIT_PROXY.rstrip("/") + "/" if GIT_PROXY else ""
        remote = (url + "https://github.com/comfyanonymous/ComfyUI.git") if url else \
                 "https://github.com/comfyanonymous/ComfyUI.git"
        print("clone:", remote)
        subprocess.check_call(["git", "clone", "--depth", "1", "--branch", COMFY_TAG,
                               remote, str(tmp)])
        shutil.copytree(tmp, COMFY_DIR, dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns("models"))
        shutil.rmtree(tmp)
    else:
        print("ComfyUI 已存在，跳过 clone")

    req = COMFY_DIR / "requirements.txt"
    keep = []
    skipped = []
    for line in req.read_text(encoding="utf-8").splitlines():
        s = line.strip()
        if not s or s.startswith("#"):
            continue
        if re.match(r"^(torch|torchvision|torchaudio|numpy|triton)([=<>!~\[]|$)|xformers", s, re.I):
            skipped.append(s)
        else:
            keep.append(s)
    filtered = ROOT / "requirements.filtered.txt"
    filtered.write_text("\n".join(keep) + "\n", encoding="utf-8")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "-r", str(filtered), "cryptography>=42"])
    print("已跳过（保护预装 GPU 环境）:", skipped or "无")
    (COMFY_DIR / "custom_nodes").mkdir(exist_ok=True)
    print("ComfyUI 源码:", COMFY_DIR)

In [ ]:
# ---------- 11. 安装自定义节点 ----------
NODES = [
    ("city96/ComfyUI-GGUF", ""),
    ("Kosinkadink/ComfyUI-VideoHelperSuite", ""),
    ("Fannovel16/ComfyUI-Frame-Interpolation", ""),
    ("christian-byrne/audio-separation-nodes-comfyui", ""),
    ("matlowai/ComfyUI-MAINodes", ""),
    ("kijai/ComfyUI-KJNodes", ""),
    ("cubiq/ComfyUI_essentials", ""),
    ("ltdrdata/ComfyUI-Manager", ""),
    ("ethanfel/ComfyUI-PlagueKind-Nodes-only-sparse", ""),
]
KEY_NODES = ["MiniMaxH3ImageToVideo", "H3SLAAttention", "H3JerkOracle", "H3TimeSmear",
             "H3DyRoPE", "H3V2VInit", "H3ExactRecover", "MiniMaxChunkFeedForward"]

def git_url(repo):
    full = "https://github.com/" + repo + ".git"
    return (GIT_PROXY.rstrip("/") + "/" + full) if GIT_PROXY else full

def install_req(folder):
    r = folder / "requirements.txt"
    if not r.exists():
        return
    keep, skipped = [], []
    for line in r.read_text(encoding="utf-8", errors="ignore").splitlines():
        s = line.strip()
        if not s or s.startswith("#"):
            continue
        if re.match(r"^(torch|torchvision|torchaudio|numpy|triton)([=<>!~\[]|$)|xformers", s, re.I):
            skipped.append(s)
        else:
            keep.append(s)
    if skipped:
        print("   过滤:", skipped)
    if keep:
        f = folder / "_req_filtered.txt"
        f.write_text("\n".join(keep) + "\n", encoding="utf-8")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(f)])

if INSTALL_COMFYUI:
    nodes_root = COMFY_DIR / "custom_nodes"
    nodes_root.mkdir(parents=True, exist_ok=True)
    for repo, sub in NODES:
        dst = nodes_root / repo.split("/")[-1]
        if dst.exists():
            print("  SKIP", dst.name)
            continue
        try:
            subprocess.check_call(["git", "clone", "--depth", "1", git_url(repo), str(dst)],
                                  stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            install_req(dst)
            print("  OK  ", dst.name)
        except Exception as e:
            print("  FAIL", repo, type(e).__name__, "-> GitHub 直连不通时设 GIT_PROXY")

    # 自检：工作流点名的节点是否都有提供方
    have = []
    for p in nodes_root.rglob("*.py"):
        try:
            t = p.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            continue
        for k in KEY_NODES:
            if ('"' + k + '"') in t and k not in have:
                have.append(k)
    miss = [k for k in KEY_NODES if k not in have]
    print("关键节点:", "全部存在" if not miss else ("缺 " + str(miss)))
    try:
        import triton
        print("Triton 可用:", triton.__version__)
    except ImportError:
        print("⚠ Triton 未安装：SLA 节点不会注册，需 pip install triton")
else:
    print("INSTALL_COMFYUI=False，跳过插件")


In [ ]:
# ---------- 12. 启动 ComfyUI + 访问地址 ----------
def _port_open(port):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{port}/", timeout=2)
        return True
    except Exception:
        return False

_missing = [v for v in ("INSTALL_COMFYUI", "START_COMFYUI", "COMFY_DIR",
                         "COMFY_PORT", "ROOT") if v not in globals()]
if _missing:
    raise RuntimeError("缺少配置 " + ", ".join(_missing) +
                       " —— 请先运行第 3 单元（配置）再启动")
vram_gb = globals().get("VRAM_GB", 0.0)


def resolve_vram_tier(vram_gb, override="auto"):
    """按【单卡】显存判档；探测失败取最保守的 low，避免漏掉 --novram。"""
    if override in ("low", "mid", "high"):
        return override
    if not vram_gb or vram_gb <= 0:
        return "low"
    if vram_gb < 24:
        return "low"
    if vram_gb < 48:
        return "mid"
    return "high"


_tier = "high" if GPU_VENDOR == "amd" else resolve_vram_tier(vram_gb, globals().get("VRAM_TIER", "auto"))
_gpu_count = int(globals().get("GPU_COUNT", 1))
_gpu_id = int(globals().get("COMFY_GPU_ID", 0))
print(f"显存判档   : GPU0 单卡 {vram_gb:.1f} GB -> {_tier} 档"
      + (f"  | 共 {_gpu_count} 卡，合计不参与判档" if _gpu_count > 1 else ""))

if START_COMFYUI and INSTALL_COMFYUI:
    if not (COMFY_DIR / "main.py").exists():
        raise RuntimeError("ComfyUI 未安装，请先跑第 10 单元")
    if _port_open(COMFY_PORT):
        print("ComfyUI 已在运行，跳过启动")
    else:
        args = [sys.executable, "main.py",
                "--listen", "0.0.0.0", "--port", str(COMFY_PORT),
                "--disable-auto-launch"]
        # v0.45.3 默认开 DynamicVRAM，--lowvram 在此模式下不起作用
        if GPU_VENDOR == "amd":
            # AMD/ROCm：不加 CUDA 专用的 --enable-triton-backend，改用 fp32 兜底
            args += ["--cache-none", "--force-fp32"]
            env = os.environ.copy()
            env["PYTHONUNBUFFERED"] = "1"
            env["HIP_VISIBLE_DEVICES"] = str(_gpu_id)
            env.pop("CUDA_VISIBLE_DEVICES", None)
        else:
            # NVIDIA/CUDA：保持原有参数
            args += ["--enable-triton-backend"]
            args += ["--cache-none"]
            args += ["--novram"] if _tier == "low" else ["--reserve-vram", "1"]
            env = os.environ.copy()
            env["PYTHONUNBUFFERED"] = "1"
            env["CUDA_VISIBLE_DEVICES"] = str(_gpu_id)
        args += COMFY_EXTRA_ARGS
        log = open(ROOT / "comfyui.log", "w")
        proc = subprocess.Popen(args, cwd=str(COMFY_DIR), stdout=log,
                                stderr=subprocess.STDOUT, env=env)
        print("ComfyUI PID:", proc.pid, "| 参数:", " ".join(args[2:]))
        ready = False
        for _ in range(90):
            if proc.poll() is not None:
                break
            if _port_open(COMFY_PORT):
                ready = True
                break
            time.sleep(2)
        print("ComfyUI 就绪:", ready, "| 日志:", ROOT / "comfyui.log")
        if not ready:
            print("\n".join((ROOT / "comfyui.log").read_text(errors="ignore").splitlines()[-40:]))

    print("\n===== 访问方式 =====")
    print(f"1) DSW / JupyterLab 端口代理：把地址栏 /lab 改成 /proxy/{COMFY_PORT}/ ")
    print("   （魔搭免费 Notebook 网关不支持自定义端口，会 302 报错 -30，请用方式 2）")
    print(f"2) 本机容器：http://127.0.0.1:{COMFY_PORT}/")

    if START_TUNNEL:
        if "cf_proc" in globals() and cf_proc.poll() is None and "public_url" in globals() and public_url:
            print("3) 隧道已在运行:", public_url)
        else:
            cf = ROOT / "cloudflared"
            if not cf.exists():
                try:
                    urllib.request.urlretrieve(
                        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                        str(cf))
                    cf.chmod(0o755)
                except Exception as e:
                    print("cloudflared 下载失败:", type(e).__name__, "-> 请手动上传二进制到", cf)
            if cf.exists():
                cf_log = open(ROOT / "cloudflared.log", "w")
                cf_proc = subprocess.Popen(
                    [str(cf), "tunnel", "--url", f"http://127.0.0.1:{COMFY_PORT}",
                     "--protocol", TUNNEL_PROTOCOL, "--no-autoupdate"],
                    stdout=cf_log, stderr=subprocess.STDOUT)
                public_url = None
                for _ in range(60):
                    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com",
                                  (ROOT / "cloudflared.log").read_text(errors="ignore"))
                    if m:
                        public_url = m.group(0)
                        break
                    time.sleep(1)
                if public_url:
                    print("3) 公网地址（推荐）:", public_url)
                else:
                    print("未拿到隧道地址，cloudflared.log 末尾：")
                    print("\n".join((ROOT / "cloudflared.log").read_text(errors="ignore").splitlines()[-30:]))
else:
    print("START_COMFYUI=False 或 INSTALL_COMFYUI=False，跳过启动")

### 启动之后 / 常见坑

- 打开界面后，把 `/mnt/workspace/minimax_h3/output/workflows/` 里的 json 拖进去；用越狱编码器就拖 `workflows_heretic/`。**A10 24 GB 优先用 `workflows/lowvram/` 那套**，并把分辨率降到 0.3–0.7 MP、帧数 ≤124。
- **装了新模型必须重启 ComfyUI**，它只在启动时扫描 `models/`，刷新网页无效。
- 报 "missing node"：看 `/mnt/workspace/minimax_h3/comfyui.log`，多半是 Triton 没装导致 SLA / 量化节点静默不注册，或插件依赖被过滤后少装了什么，按日志补 `pip install`。
- OOM：先降分辨率/帧数，再试 `--novram`；仍不行就设 `LIGHT_MODEL="w4a8"`（12.5 GB）或 `"w6a8"`（16 GB）并改用 **ComfyUI 内置 H3 模板**——本套 MATLOWAI 工作流写死了 21 GB fused 文件名。
- `--fast-disk` 只在 `/mnt/workspace` 是高速本地盘时才有意义；它是 NAS 网络盘，默认没加，可自行试 `COMFY_EXTRA_ARGS=["--fast-disk"]` 对比。
- 工作流 `03_derope_dyrope_pb3049_warm375` 额外引用 `minimax_h3_temporal_expansion_step375_r16.safetensors`，魔搭站内未确认到公开直链，用 03 前需自行补。
- 免费实例空闲约 1 小时自动关、单次约 8 小时强制停；所有权重都在 `/mnt/workspace`，重开实例后第 5 单元会显示 `REUSE [cache]`，**零下载**。
- 模型许可为 MiniMax H3 Community License，商用有地域与营收限制，部署前自行确认。